# 迪士尼游客数据分析 Notebook

本Notebook对迪士尼游客数据进行完整的清洗、分析和可视化。

## 目录
1. 数据加载与预览
2. 数据清洗
3. 探索性数据分析 (EDA)
4. 特征工程
5. 数据可视化
6. 数据保存

## 1. 数据加载与预览

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体和图表样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("库导入成功！")

In [ ]:
# 加载数据
df = pd.read_csv('../data/raw/disney_attendance.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"数据集形状: {df.shape}")
print(f"\n数据集包含 {len(df)} 条记录，{len(df.columns)} 个特征")

In [ ]:
# 查看前几行数据
df.head(10)

In [ ]:
# 查看数据类型
print("数据类型:")
print(df.dtypes)

In [ ]:
# 数据基本统计信息
df.describe()

## 2. 数据清洗

In [ ]:
# 检查缺失值
print("缺失值统计:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "没有缺失值")

In [ ]:
# 检查重复值
duplicates = df.duplicated().sum()
print(f"重复记录数: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"已删除 {duplicates} 条重复记录")

In [ ]:
# 检查异常值 - 使用IQR方法
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

outliers, lower, upper = detect_outliers_iqr(df, 'visitors')
print(f"游客数量异常值统计:")
print(f"  - 下界: {lower:.0f}")
print(f"  - 上界: {upper:.0f}")
print(f"  - 异常值数量: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

In [ ]:
# 查看异常值详情（可能是特殊事件导致的极端值）
if len(outliers) > 0:
    print("异常值日期分布:")
    print(outliers[['date', 'visitors', 'is_holiday', 'is_peak_season']].head(20))

In [ ]:
# 数据类型转换和验证
# 确保日期格式正确
df['date'] = pd.to_datetime(df['date'])

# 确保数值列为整数
int_columns = ['visitors', 'temperature', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year']
for col in int_columns:
    if col in df.columns:
        df[col] = df[col].astype(int)

# 确保布尔列为整数类型
bool_columns = ['is_weekend', 'is_holiday', 'is_peak_season', 'is_school_holiday', 'is_rainy']
for col in bool_columns:
    if col in df.columns:
        df[col] = df[col].astype(int)

print("数据类型转换完成！")

## 3. 探索性数据分析 (EDA)

In [ ]:
# 按年份统计游客数量
yearly_stats = df.groupby('year')['visitors'].agg(['mean', 'median', 'std', 'min', 'max', 'sum'])
yearly_stats.columns = ['平均值', '中位数', '标准差', '最小值', '最大值', '总游客数']
print("年度游客统计:")
print(yearly_stats.round(0))

In [ ]:
# 按月份统计游客数量
monthly_stats = df.groupby('month')['visitors'].agg(['mean', 'median', 'std']).round(0)
monthly_stats.columns = ['平均游客数', '中位数', '标准差']
month_names = ['1月', '2月', '3月', '4月', '5月', '6月', '7月', '8月', '9月', '10月', '11月', '12月']
monthly_stats.index = month_names
print("\n月度游客统计:")
print(monthly_stats)

In [ ]:
# 按星期统计游客数量
weekday_stats = df.groupby('day_of_week')['visitors'].agg(['mean', 'median', 'std']).round(0)
weekday_stats.columns = ['平均游客数', '中位数', '标准差']
weekday_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
weekday_stats.index = weekday_names
print("\n星期游客统计:")
print(weekday_stats)

In [ ]:
# 节假日 vs 非节假日对比
holiday_comparison = df.groupby('is_holiday')['visitors'].agg(['mean', 'median', 'count'])
holiday_comparison.index = ['非节假日', '节假日']
holiday_comparison.columns = ['平均游客数', '中位数', '天数']
print("\n节假日 vs 非节假日:")
print(holiday_comparison.round(0))

In [ ]:
# 周末 vs 工作日对比
weekend_comparison = df.groupby('is_weekend')['visitors'].agg(['mean', 'median', 'count'])
weekend_comparison.index = ['工作日', '周末']
weekend_comparison.columns = ['平均游客数', '中位数', '天数']
print("\n周末 vs 工作日:")
print(weekend_comparison.round(0))

In [ ]:
# 相关性分析
numeric_columns = ['visitors', 'month', 'day_of_week', 'is_weekend', 'is_holiday', 
                   'is_peak_season', 'is_school_holiday', 'temperature', 'rain_probability', 'is_rainy']
correlation_matrix = df[numeric_columns].corr()

print("\n与游客数量的相关性:")
visitors_corr = correlation_matrix['visitors'].sort_values(ascending=False)
print(visitors_corr)

## 4. 特征工程

In [ ]:
# 添加更多时间特征
df['quarter'] = df['date'].dt.quarter
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

# 添加季节特征
def get_season(month):
    if month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return 'Winter'

df['season'] = df['month'].apply(get_season)

# 添加滞后特征（前一天的游客数）
df['visitors_lag1'] = df['visitors'].shift(1)
df['visitors_lag7'] = df['visitors'].shift(7)  # 上周同一天

# 添加移动平均特征
df['visitors_ma7'] = df['visitors'].rolling(window=7).mean()
df['visitors_ma30'] = df['visitors'].rolling(window=30).mean()

# 填充NaN值
df['visitors_lag1'] = df['visitors_lag1'].fillna(df['visitors'].mean())
df['visitors_lag7'] = df['visitors_lag7'].fillna(df['visitors'].mean())
df['visitors_ma7'] = df['visitors_ma7'].fillna(df['visitors'].mean())
df['visitors_ma30'] = df['visitors_ma30'].fillna(df['visitors'].mean())

print("特征工程完成！")
print(f"新增特征: quarter, is_month_start, is_month_end, season, visitors_lag1, visitors_lag7, visitors_ma7, visitors_ma30")

In [ ]:
# 查看新增特征后的数据
df.head()

## 5. 数据可视化

In [ ]:
# 设置图表大小
fig_size = (14, 6)

In [ ]:
# 1. 游客数量时间序列图
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['date'], df['visitors'], alpha=0.5, linewidth=0.5, label='每日游客')
ax.plot(df['date'], df['visitors_ma30'], color='red', linewidth=2, label='30日移动平均')
ax.set_xlabel('日期', fontsize=12)
ax.set_ylabel('游客数量', fontsize=12)
ax.set_title('迪士尼每日游客数量趋势 (2018-2024)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../images/visitors_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. 年度游客数量箱线图
fig, ax = plt.subplots(figsize=(12, 6))
df.boxplot(column='visitors', by='year', ax=ax)
ax.set_xlabel('年份', fontsize=12)
ax.set_ylabel('游客数量', fontsize=12)
ax.set_title('年度游客数量分布', fontsize=14, fontweight='bold')
plt.suptitle('')  # 移除自动生成的标题
plt.tight_layout()
plt.savefig('../images/yearly_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3. 月度游客数量热力图
monthly_avg = df.groupby(['year', 'month'])['visitors'].mean().unstack()
monthly_avg.columns = month_names

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(monthly_avg, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, 
            cbar_kws={'label': '平均游客数'})
ax.set_xlabel('月份', fontsize=12)
ax.set_ylabel('年份', fontsize=12)
ax.set_title('月度平均游客数量热力图', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/monthly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4. 星期游客数量分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱线图
df.boxplot(column='visitors', by='day_of_week', ax=axes[0])
axes[0].set_xticklabels(weekday_names)
axes[0].set_xlabel('星期', fontsize=12)
axes[0].set_ylabel('游客数量', fontsize=12)
axes[0].set_title('星期游客数量分布')

# 条形图
weekday_means = df.groupby('day_of_week')['visitors'].mean()
colors = plt.cm.viridis(np.linspace(0.2, 0.8, 7))
axes[1].bar(weekday_names, weekday_means, color=colors)
axes[1].set_xlabel('星期', fontsize=12)
axes[1].set_ylabel('平均游客数量', fontsize=12)
axes[1].set_title('星期平均游客数量')

plt.suptitle('')
plt.tight_layout()
plt.savefig('../images/weekday_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5. 季节性分析
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 月度平均游客
monthly_means = df.groupby('month')['visitors'].mean()
colors = plt.cm.coolwarm(np.linspace(0.1, 0.9, 12))
axes[0].bar(month_names, monthly_means, color=colors)
axes[0].set_xlabel('月份', fontsize=12)
axes[0].set_ylabel('平均游客数量', fontsize=12)
axes[0].set_title('月度平均游客数量')
axes[0].tick_params(axis='x', rotation=45)

# 季节对比
season_means = df.groupby('season')['visitors'].mean().reindex(['Spring', 'Summer', 'Fall', 'Winter'])
season_colors = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']
axes[1].bar(season_means.index, season_means, color=season_colors)
axes[1].set_xlabel('季节', fontsize=12)
axes[1].set_ylabel('平均游客数量', fontsize=12)
axes[1].set_title('季节平均游客数量')

plt.tight_layout()
plt.savefig('../images/seasonal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6. 相关性热力图
fig, ax = plt.subplots(figsize=(12, 10))
correlation_matrix = df[numeric_columns].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='RdYlBu_r', center=0, 
            fmt='.2f', ax=ax, square=True)
ax.set_title('特征相关性热力图', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 7. 节假日和周末影响分析
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 节假日对比
holiday_data = [df[df['is_holiday']==0]['visitors'], df[df['is_holiday']==1]['visitors']]
bp1 = axes[0].boxplot(holiday_data, labels=['非节假日', '节假日'], patch_artist=True)
bp1['boxes'][0].set_facecolor('#3498db')
bp1['boxes'][1].set_facecolor('#e74c3c')
axes[0].set_ylabel('游客数量', fontsize=12)
axes[0].set_title('节假日 vs 非节假日游客数量')

# 周末对比
weekend_data = [df[df['is_weekend']==0]['visitors'], df[df['is_weekend']==1]['visitors']]
bp2 = axes[1].boxplot(weekend_data, labels=['工作日', '周末'], patch_artist=True)
bp2['boxes'][0].set_facecolor('#9b59b6')
bp2['boxes'][1].set_facecolor('#f1c40f')
axes[1].set_ylabel('游客数量', fontsize=12)
axes[1].set_title('工作日 vs 周末游客数量')

plt.tight_layout()
plt.savefig('../images/holiday_weekend_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8. 游客数量分布直方图
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df['visitors'], bins=50, color='#3498db', edgecolor='white', alpha=0.7)
ax.axvline(df['visitors'].mean(), color='red', linestyle='--', linewidth=2, label=f'平均值: {df["visitors"].mean():.0f}')
ax.axvline(df['visitors'].median(), color='green', linestyle='--', linewidth=2, label=f'中位数: {df["visitors"].median():.0f}')
ax.set_xlabel('游客数量', fontsize=12)
ax.set_ylabel('频数', fontsize=12)
ax.set_title('游客数量分布直方图', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../images/visitors_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 9. 温度与游客数量关系
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(df['temperature'], df['visitors'], c=df['month'], 
                     cmap='coolwarm', alpha=0.5, s=10)
plt.colorbar(scatter, label='月份')
ax.set_xlabel('温度 (°F)', fontsize=12)
ax.set_ylabel('游客数量', fontsize=12)
ax.set_title('温度与游客数量关系', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/temperature_visitors.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 10. 年度趋势对比（排除2020年疫情影响）
fig, ax = plt.subplots(figsize=(14, 6))

for year in range(2018, 2025):
    year_data = df[df['year'] == year]
    ax.plot(year_data['day_of_year'], year_data['visitors_ma7'], 
            label=str(year), alpha=0.7, linewidth=1.5)

ax.set_xlabel('一年中的第几天', fontsize=12)
ax.set_ylabel('游客数量 (7日移动平均)', fontsize=12)
ax.set_title('各年度游客趋势对比', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../images/yearly_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 数据保存

In [ ]:
# 保存处理后的数据
output_path = '../data/processed/disney_attendance_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"清洗后的数据已保存到: {output_path}")
print(f"数据形状: {df.shape}")

In [ ]:
# 数据摘要报告
print("=" * 60)
print("迪士尼游客数据分析报告摘要")
print("=" * 60)
print(f"\n数据时间范围: {df['date'].min().strftime('%Y-%m-%d')} 至 {df['date'].max().strftime('%Y-%m-%d')}")
print(f"总记录数: {len(df):,}")
print(f"\n游客数量统计:")
print(f"  - 平均值: {df['visitors'].mean():,.0f} 人/天")
print(f"  - 中位数: {df['visitors'].median():,.0f} 人/天")
print(f"  - 最小值: {df['visitors'].min():,} 人/天")
print(f"  - 最大值: {df['visitors'].max():,} 人/天")
print(f"  - 标准差: {df['visitors'].std():,.0f}")
print(f"\n关键发现:")
print(f"  1. 高峰月份: {monthly_stats['平均游客数'].idxmax()} (平均 {monthly_stats['平均游客数'].max():,.0f} 人)")
print(f"  2. 低谷月份: {monthly_stats['平均游客数'].idxmin()} (平均 {monthly_stats['平均游客数'].min():,.0f} 人)")
print(f"  3. 周末平均游客: {df[df['is_weekend']==1]['visitors'].mean():,.0f} 人")
print(f"  4. 工作日平均游客: {df[df['is_weekend']==0]['visitors'].mean():,.0f} 人")
print(f"  5. 节假日平均游客: {df[df['is_holiday']==1]['visitors'].mean():,.0f} 人")
print(f"  6. 非节假日平均游客: {df[df['is_holiday']==0]['visitors'].mean():,.0f} 人")
print("\n" + "=" * 60)